# Improvements Introduced in Mistral 7B

## 1. Sliding Window Attention (SWA)

Traditional Transformers use full attention where every token attends to all previous tokens.

This creates quadratic complexity:

```text
O(n²)
```

Mistral introduces Sliding Window Attention, where each token attends only to a fixed recent window of tokens.

Example:

- Window size = 4096
- Token 10000 attends only to recent 4096 tokens

instead of attending to the entire sequence.

Benefits:
- lower attention computation
- reduced memory usage
- faster inference
- efficient long-context handling

This works because nearby tokens are usually more important than very old tokens.

---

## 2. Rolling Buffer KV Cache

Standard Transformers store KV cache for all previous tokens, causing memory to grow with sequence length.

Mistral uses a rolling buffer cache:
- fixed cache size = W
- old cache entries are overwritten using:

```text
i mod W
```

### Small Example

Suppose:

```text
W = 3
```

The cache has only 3 slots.

### Step 1: Store Token 1

```text
1 mod 3 = 1
```

| Slot | Content |
|---|---|
| 0 | empty |
| 1 | token 1 |
| 2 | empty |

---

### Step 2: Store Token 2

```text
2 mod 3 = 2
```

| Slot | Content |
|---|---|
| 0 | empty |
| 1 | token 1 |
| 2 | token 2 |

---

### Step 3: Store Token 3

```text
3 mod 3 = 0
```

| Slot | Content |
|---|---|
| 0 | token 3 |
| 1 | token 1 |
| 2 | token 2 |

---

### Step 4: Store Token 4

```text
4 mod 3 = 1
```

Slot 1 gets overwritten.

| Slot | Content |
|---|---|
| 0 | token 3 |
| 1 | token 4 |
| 2 | token 2 |

Token 1 is removed from cache.

---

### Cache Complexity Comparison

#### Standard Transformer Cache

Without rolling buffer:

```text
O(nLd)
```

because all previous token KV states are stored.

Where:
- n = sequence length
- L = number of layers
- d = hidden dimension

---

#### Rolling Buffer Cache

With Sliding Window Attention:

only the most recent:

```text
W
```

tokens remain in cache.

So cache memory becomes:

```text
O(WLd)
```

instead of:

```text
O(nLd)
```

No matter how long generation continues.

---

### Real Example

| Sequence Length | Window Size |
|---|---|
| 32K tokens | 4K window |

Memory reduction:

```text
32000 / 4000 = 8×
```

So the model stores only recent 4K token KV states instead of all 32K tokens.

Benefits:
- fixed memory usage
- efficient long-sequence inference
- reduced GPU memory consumption

---

## 3. Grouped Query Attention (GQA)

Traditional multi-head attention stores separate:
- Query (Q)
- Key (K)
- Value (V)

for every attention head.

### Standard Multi-Head Attention

Suppose we have:

```text
8 attention heads
```

Traditional attention:

| Query Head | Key Head | Value Head |
|---|---|---|
| Q1 | K1 | V1 |
| Q2 | K2 | V2 |
| Q3 | K3 | V3 |
| Q4 | K4 | V4 |
| Q5 | K5 | V5 |
| Q6 | K6 | V6 |
| Q7 | K7 | V7 |
| Q8 | K8 | V8 |

This means:
- 8 K caches
- 8 V caches

must be stored.

---

### GQA Idea

Mistral keeps:
- 8 Query heads

but reduces:
- KV heads to only 2

Example:

| Query Heads | Shared KV Head |
|---|---|
| Q1, Q2, Q3, Q4 | KV1 |
| Q5, Q6, Q7, Q8 | KV2 |

Now:
- Q1–Q4 share one KV memory
- Q5–Q8 share another KV memory

Instead of:

```text
8 K heads + 8 V heads
```

the model stores only:

```text
2 K heads + 2 V heads
```

Memory reduction:

```text
8 / 2 = 4× smaller KV cache
```

Benefits:
- smaller KV cache
- lower memory bandwidth
- faster inference
- improved deployment efficiency

---

## 4. Pre-fill and Chunking

During generation, LLMs generate tokens one-by-one because each token depends on previous tokens.

Example:

```text
token1 → token2 → token3 → token4
```

This is called autoregressive decoding.

---

### Pre-fill Phase

The prompt is already known before generation starts.

So the model first computes:
- Key (K)
- Value (V)

for all prompt tokens and stores them in the KV cache.

This is called:

```text
Pre-fill
```

Instead of processing prompt tokens one-by-one, the model processes them in parallel.

---

### Chunking

Large prompts are expensive to process at once.

Mistral splits prompts into smaller chunks.

Example:

| Chunk | Tokens |
|---|---|
| Chunk 1 | 1–4096 |
| Chunk 2 | 4097–8192 |
| Chunk 3 | 8193–12288 |

The chunk size is usually equal to the sliding window size.

---

### How Chunk Processing Works

For each chunk, the model computes attention over:
1. Existing KV cache
2. Tokens inside the current chunk

Example:

When processing Chunk 2:
- Chunk 2 attends to Chunk 1 cache
- Chunk 2 also attends to earlier tokens inside Chunk 2

---

### Why This Helps

Instead of computing one huge attention matrix:

```text
O(32000²)
```

the model processes smaller chunks like:

```text
O(4096²)
```

Benefits:
- lower memory usage
- efficient GPU utilization
- scalable long-context inference

GQA further improves chunked inference efficiency by reducing KV cache cost.

---

## 5. RMSNorm

Older GPT models used LayerNorm.

Mistral uses RMSNorm:
- simpler normalization
- uses root mean square only

Benefits:
- lower computation cost
- improved training stability
- better efficiency

---

## 6. SwiGLU Feedforward Activation

Older models commonly used GELU activation.

Mistral uses SwiGLU activation in feedforward layers.

Benefits:
- better gradient flow
- improved parameter efficiency
- stronger learning capability

This allows smaller models to achieve stronger performance.

---

## 7. Better Parameter Efficiency

Mistral demonstrated that:
- architecture quality matters more than simply increasing parameter count

Despite having only 7B parameters, Mistral 7B performed competitively against much larger models.

Benefits:
- lower hardware requirements
- faster deployment
- smaller GPU memory usage
- strong performance-per-parameter ratio

---

# Main Contribution of Mistral

Mistral 7B showed that:

```text
Smarter architecture + efficient attention
>
Simply increasing model size
```

This became a major shift in modern open-source LLM design.


# GPT vs LLaMA vs Mistral

| Feature | GPT | LLaMA | Mistral |
|---|---|---|---|
| Organization | OpenAI | Meta | Mistral AI |
| Main Goal | powerful general LLMs | efficient open LLMs | optimized efficient inference |
| Architecture | decoder-only Transformer | optimized decoder-only Transformer | optimized decoder-only Transformer |
| Training Focus | large-scale scaling | efficient scaling + better data | efficient attention + memory optimization |
| Attention Type | full causal attention | full causal attention | Sliding Window Attention |
| Positional Embedding | learned absolute → RoPE in newer models | RoPE | RoPE |
| Normalization | LayerNorm | RMSNorm | RMSNorm |
| Activation | GELU | SwiGLU | SwiGLU |
| KV Cache | standard growing cache | standard cache | rolling buffer cache |
| Query Attention | Multi-Head Attention | MHA / GQA in newer versions | GQA |
| Long Context Efficiency | expensive | improved | highly optimized |
| Inference Speed | slower for large contexts | efficient | faster |
| Memory Usage | high | reduced | highly reduced |
| Open Source | mostly closed | open weights | open weights |
| Main Innovation | generative pretraining + few-shot learning | efficient scaling and training | optimized attention and inference |

---

# 1. GPT

GPT models introduced:
- generative pretraining
- next-token prediction
- large-scale language modeling

Main idea:

```text
Bigger models + more data = stronger intelligence
```

GPT-3 especially introduced:
- powerful few-shot learning
- in-context learning

Example:

```text
English → French examples
```

The model learns tasks directly from prompts without weight updates.

## GPT Strengths

- strong reasoning
- strong language generation
- few-shot learning
- emergent abilities

## GPT Weaknesses

- very expensive
- huge GPU memory requirements
- quadratic attention cost

---

# 2. LLaMA

LLaMA focused on:

- efficient training
- better scaling laws
- high-quality training data
- parameter efficiency

Main idea:

```text
Train smarter instead of only making models larger
```

LLaMA showed that:
- 7B–13B models can rival much larger older models

because:
- training data was cleaner
- token usage was optimized
- scaling laws were improved

## LLaMA Strengths

- smaller but powerful
- efficient training
- strong open-source ecosystem
- lower hardware requirements


## LLaMA Weaknesses

Still uses:
- full attention
- growing KV cache

So long-context inference remains expensive.


# 3. Mistral

Mistral focused on:

- inference optimization
- memory efficiency
- efficient attention
- long-context deployment

Main idea:

```text
Run smarter
```

Instead of only improving training.
## How Mistral Optimized Inference

Inference means:

```text
Using the trained model to generate tokens
```

Example:

```text
Prompt → model → generated response
```

Inference becomes expensive because:
- attention computation is large
- KV cache grows
- GPU memory bandwidth becomes a bottleneck

Mistral optimized inference by reducing:
- attention cost
- cache size
- memory movement
- long-context computation

---



# Core Philosophy Comparison

| Model | Main Philosophy |
|---|---|
| GPT | scale bigger |
| LLaMA | train smarter |
| Mistral | run smarter |

---

# Evolution Path

| Model Family | Main Innovation |
|---|---|
| GPT | generative pretraining |
| GPT-3 | few-shot learning |
| LLaMA | efficient open scaling |
| Mistral | optimized inference and attention |

---

# Simple Analogy

## GPT

```text
Huge powerful engine
```

---

## LLaMA

```text
Smaller engine trained efficiently
```

---

## Mistral

```text
Efficient engine + optimized transmission + optimized memory system
```

---


# Final Summary

GPT proved that scaling large language models creates powerful emergent abilities.

LLaMA showed that smaller models can achieve strong performance through better training and efficient scaling.

Mistral improved inference efficiency further using optimized attention, KV cache management, and long-context processing.

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1. Grouped Query Attention + Sliding Window

In [9]:
class MistralStyleAttention(nn.Module):
    def __init__(self, d_model=128, n_q_heads=8, n_kv_heads=2, window_size=4):
        super().__init__()

        self.d_model = d_model
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_q_heads
        self.window_size = window_size

        self.q_proj = nn.Linear(d_model, n_q_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(d_model, d_model)

        self.group_size = n_q_heads // n_kv_heads

    def repeat_kv(self, x):
        # x: [B, kv_heads, T, head_dim]
        # repeat KV heads so they match query heads
        return x.repeat_interleave(self.group_size, dim=1)

    def forward(self, x):
        B, T, _ = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(B, T, self.n_q_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # GQA: share fewer KV heads across many Q heads
        K = self.repeat_kv(K)
        V = self.repeat_kv(V)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # causal mask: token cannot see future tokens
        causal_mask = torch.tril(torch.ones(T, T, device=x.device))

        # sliding window mask: token sees only recent W tokens
        window_mask = torch.zeros(T, T, device=x.device)
        for i in range(T):
            start = max(0, i - self.window_size + 1)
            window_mask[i, start:i+1] = 1

        mask = causal_mask * window_mask

        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)

        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.out_proj(out)

# 2. Test Sliding Window + GQA

In [10]:
x = torch.randn(1, 10, 128)

attn = MistralStyleAttention(
    d_model=128,
    n_q_heads=8,
    n_kv_heads=2,
    window_size=4
)

y = attn(x)

print(y.shape)

torch.Size([1, 10, 128])


# 3. Rolling KV Cache

In [11]:
class RollingKVCache:
    def __init__(self, window_size, n_kv_heads, head_dim):
        self.W = window_size
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim

        self.K_cache = torch.zeros(1, n_kv_heads, window_size, head_dim)
        self.V_cache = torch.zeros(1, n_kv_heads, window_size, head_dim)

    def update(self, timestep, K_new, V_new):
        # position in circular buffer
        pos = timestep % self.W

        self.K_cache[:, :, pos, :] = K_new
        self.V_cache[:, :, pos, :] = V_new

        return self.K_cache, self.V_cache

In [12]:
cache = RollingKVCache(
    window_size=3,
    n_kv_heads=2,
    head_dim=16
)

for t in range(6):
    K_new = torch.randn(1, 2, 16)
    V_new = torch.randn(1, 2, 16)

    K_cache, V_cache = cache.update(t, K_new, V_new)

    print(f"Token {t} stored at position {t % 3}")

Token 0 stored at position 0
Token 1 stored at position 1
Token 2 stored at position 2
Token 3 stored at position 0
Token 4 stored at position 1
Token 5 stored at position 2


# 4. Chunked Pre-fill

In [13]:
def chunked_prefill(model, prompt_embeddings, chunk_size=4):
    B, T, D = prompt_embeddings.shape

    outputs = []

    for start in range(0, T, chunk_size):
        end = min(start + chunk_size, T)

        chunk = prompt_embeddings[:, start:end, :]

        print(f"Processing chunk {start} to {end}")

        chunk_output = model(chunk)
        outputs.append(chunk_output)

    return torch.cat(outputs, dim=1)

In [14]:
prompt = torch.randn(1, 16, 128)

output = chunked_prefill(
    model=attn,
    prompt_embeddings=prompt,
    chunk_size=4
)

print(output.shape)

Processing chunk 0 to 4
Processing chunk 4 to 8
Processing chunk 8 to 12
Processing chunk 12 to 16
torch.Size([1, 16, 128])


In [15]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return self.weight * x
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim)
        self.w2 = nn.Linear(dim, hidden_dim)
        self.w3 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))        

In [16]:
class TinyMistralAttention(nn.Module):
    def __init__(self, dim=128, n_q_heads=8, n_kv_heads=2, window_size=16):
        super().__init__()

        self.dim = dim
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_q_heads
        self.window_size = window_size

        self.q_proj = nn.Linear(dim, n_q_heads * self.head_dim)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim)
        self.o_proj = nn.Linear(dim, dim)

        self.group_size = n_q_heads // n_kv_heads

    def repeat_kv(self, x):
        return x.repeat_interleave(self.group_size, dim=1)

    def forward(self, x):
        B, T, C = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(B, T, self.n_q_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        K = self.repeat_kv(K)
        V = self.repeat_kv(V)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        causal_mask = torch.tril(torch.ones(T, T, device=x.device))

        window_mask = torch.zeros(T, T, device=x.device)
        for i in range(T):
            start = max(0, i - self.window_size + 1)
            window_mask[i, start:i + 1] = 1

        mask = causal_mask * window_mask
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = attn @ V

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)

In [17]:
class TinyMistralBlock(nn.Module):
    def __init__(self, dim=128, n_q_heads=8, n_kv_heads=2, window_size=16, mlp_hidden=256):
        super().__init__()

        self.attn_norm = RMSNorm(dim)
        self.attn = TinyMistralAttention(
            dim=dim,
            n_q_heads=n_q_heads,
            n_kv_heads=n_kv_heads,
            window_size=window_size
        )

        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, mlp_hidden)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

In [18]:
class TinyMistralLM(nn.Module):
    def __init__(
        self,
        vocab_size=1000,
        dim=128,
        n_layers=4,
        n_q_heads=8,
        n_kv_heads=2,
        window_size=16,
        max_seq_len=128
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        self.blocks = nn.ModuleList([
            TinyMistralBlock(
                dim=dim,
                n_q_heads=n_q_heads,
                n_kv_heads=n_kv_heads,
                window_size=window_size
            )
            for _ in range(n_layers)
        ])

        self.norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, input_ids):
        B, T = input_ids.shape

        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)

        x = self.token_emb(input_ids) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        logits = self.lm_head(x)

        return logits

In [22]:
# -----------------------------
# Tiny training example
# -----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyMistralLM(
    vocab_size=1000,
    dim=128,
    n_layers=4,
    n_q_heads=8,
    n_kv_heads=2,
    window_size=16,
    max_seq_len=128
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# fake training data
# batch_size = 8, sequence_length = 64
input_ids = torch.randint(0, 1000, (8, 64)).to(device)

# next-token prediction
x = input_ids[:, :-1]
y = input_ids[:, 1:]

for step in range(200):
    logits = model(x)

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

step 0 | loss 7.0893
step 20 | loss 5.7955
step 40 | loss 4.3305
step 60 | loss 2.6514
step 80 | loss 1.1997
step 100 | loss 0.4774
step 120 | loss 0.2381
step 140 | loss 0.1509
step 160 | loss 0.1093
step 180 | loss 0.0851


In [ ]:

text = "hello world " * 1000

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

model = TinyMistralLM(
    vocab_size=len(chars),
    dim=128,
    n_layers=4,
    n_q_heads=8,
    n_kv_heads=2,
    window_size=16,
    max_seq_len=128
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

block_size = 64
batch_size = 16

def get_batch():
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

for step in range(1000):
    x, y = get_batch()

    logits = model(x)

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

In [ ]:
@torch.no_grad()
def generate_text(model, start_text, max_new_tokens=100):
    model.eval()

    input_ids = torch.tensor(
        [[stoi[ch] for ch in start_text]],
        dtype=torch.long
    ).to(device)

    for _ in range(max_new_tokens):
        input_cond = input_ids[:, -block_size:]

        logits = model(input_cond)
        next_logits = logits[:, -1, :]

        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_id], dim=1)

    output = "".join([itos[i] for i in input_ids[0].tolist()])
    return output
print(generate_text(model, "hello", max_new_tokens=200))    

In [ ]:
# For deterministic output:
@torch.no_grad()
def generate_text_greedy(model, start_text, max_new_tokens=100):
    model.eval()

    input_ids = torch.tensor(
        [[stoi[ch] for ch in start_text]],
        dtype=torch.long
    ).to(device)

    for _ in range(max_new_tokens):
        input_cond = input_ids[:, -block_size:]

        logits = model(input_cond)
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        input_ids = torch.cat([input_ids, next_id], dim=1)

    return "".join([itos[i] for i in input_ids[0].tolist()])
print(generate_text_greedy(model, "hello", max_new_tokens=200))    

# Test Tiny Mistral

In [23]:
model = TinyMistralLM(
    vocab_size=1000,
    dim=128,
    n_layers=4,
    n_q_heads=8,
    n_kv_heads=2,
    window_size=16,
    max_seq_len=128
)

input_ids = torch.randint(0, 1000, (2, 32))

logits = model(input_ids)

print(logits.shape)

torch.Size([2, 32, 1000])


# Tiny generation loop

In [24]:
@torch.no_grad()
def generate(model, input_ids, max_new_tokens=20):
    model.eval()

    for _ in range(max_new_tokens):
        logits = model(input_ids)

        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return input_ids

In [25]:
start = torch.tensor([[1, 5, 9, 12]])

generated = generate(model, start, max_new_tokens=10)

print(generated)

tensor([[  1,   5,   9,  12,  56, 490, 118, 835, 933, 301, 126, 768, 229, 333]])
